# Calibración del peso q(k,v) para candidatos en la misma fila (Fase 4)

`KeyValueExtractor._score_value_candidate` da a un candidato situado en la misma fila
que la clave (`R_→`) una bonificación `q(k,v)` sobre uno alineado debajo (`q(k,v)=1`).
El valor usado en el código, `q(k,v)=1.2`, no estaba calibrado — venía de un valor previo
(1.1) sin ninguna justificación documentada, encontrado insuficiente al evaluar el
resultado completo de `_build_form` sobre el conjunto held-out de `evaluate/` (ver
`evaluate_association.ipynb`): con `q(k,v)=1.1`, en 2 de los 10 documentos held-out un
candidato alineado debajo pero mucho más cercano (p. ej. el texto de una clave vecina)
le ganaba al candidato correcto en la misma fila, produciendo 6 asociaciones incorrectas.

Este notebook calibra `q(k,v)` **usando exclusivamente el conjunto de calibración de
Fase 4** (`data/labeled/` + `data/expected_associations/`, 25 documentos, 924 parejas
clave-valor verificadas a mano — el mismo conjunto usado para calibrar
`ROW_TOLERANCE`/`EDGE_TOLERANCE`/`COLUMN_TOLERANCE`), sin tocar el conjunto held-out de
`evaluate/`: como indica la sección de Diseño experimental, ese conjunto no debe usarse
para seleccionar parámetros, solo para evaluar el resultado final.

In [ ]:
import sys
import json
import types
from pathlib import Path

import pandas as pd

ROOT = Path().resolve()
while ROOT.name != "pdf-key-extraction":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from extract.key_value_extractor import KeyValueExtractor, MAX_NORMALIZED_DISTANCE

LABELED_DIR = ROOT / "data" / "labeled"
EXPECTED_DIR = ROOT / "data" / "expected_associations"
doc_stems = sorted(p.stem for p in EXPECTED_DIR.glob("*.json"))
print(f"Documentos de calibracion: {len(doc_stems)}")

C:\Users\Samuel\miniconda3\envs\pdf-key-extraction\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Documentos de calibracion: 25


## 1. Extractor con `q(k,v)` parametrizable

Se reutiliza `KeyValueExtractor._score_value_candidate` tal cual, solo reemplazando el
literal `1.2` por un parámetro `same_row_bonus`, para poder barrer su valor sin tocar el
código de producción.

In [ ]:
def make_extractor(same_row_bonus):
    extractor = object.__new__(KeyValueExtractor)
    extractor.FIELD_KEY_PREFIX = "FIELD_KEY_"
    extractor.FIELD_VALUE_PREFIX = "FIELD_VALUE_"
    extractor.HEADER_PREFIX = "HEADER_"
    extractor.ITEM_PREFIX = "ITEM_"
    extractor.ROW_TOLERANCE = 8
    extractor.EDGE_TOLERANCE = 8
    extractor.COLUMN_TOLERANCE = 50
    extractor.PAGE_MAX_DISTANCE = MAX_NORMALIZED_DISTANCE
    extractor.AMBIGUITY_K = 0.05

    def _score_value_candidate(self, key_bbox, value_bbox, _bonus=same_row_bonus):
        kx0, ky0, kx1, ky1 = key_bbox
        vx0, vy0, vx1, vy1 = value_bbox

        key_center = ((kx0 + kx1) / 2, (ky0 + ky1) / 2)
        value_center = ((vx0 + vx1) / 2, (vy0 + vy1) / 2)

        same_row = abs(ky0 - vy0) <= self.ROW_TOLERANCE
        to_the_right = vx0 >= kx1 - self.EDGE_TOLERANCE

        centered = abs(value_center[0] - key_center[0]) <= self.COLUMN_TOLERANCE
        horizontal_gap = max(0.0, max(kx0, vx0) - min(kx1, vx1))
        overlapping = horizontal_gap <= self.COLUMN_TOLERANCE
        aligned_below = vy0 >= ky1 - self.EDGE_TOLERANCE and (centered or overlapping)

        if same_row and to_the_right:
            tier = _bonus
        elif aligned_below:
            tier = 1
        else:
            return None

        distance = (
            (key_center[0] - value_center[0]) ** 2
            + (key_center[1] - value_center[1]) ** 2
        ) ** 0.5
        return tier * self.PAGE_MAX_DISTANCE - distance

    extractor._score_value_candidate = types.MethodType(_score_value_candidate, extractor)
    return extractor

## 2. Verdad de campo (924 parejas, 25 documentos)

Se reutiliza tal cual la carga y el chequeo de alcanzabilidad de
`calibrate_association_tolerance.ipynb` (incluye la corrección ya aplicada de IRBPNR).

In [ ]:
def load_entities(stem):
    with open(LABELED_DIR / f"{stem}.json", encoding="utf-8") as f:
        data = json.load(f)
    return [
        {"text": e["text"], "bbox": tuple(e["bbox"]), "label": e["label"], "page": e["page"]}
        for e in data
    ]


def load_expected(stem):
    with open(EXPECTED_DIR / f"{stem}.json", encoding="utf-8") as f:
        data = json.load(f)
    expected_by_bbox = {}
    expected_by_text = {}
    for item in data:
        bbox = tuple(item["key_bbox"]) if item.get("key_bbox") is not None else None
        if bbox is not None:
            expected_by_bbox[(item["page"], item["suffix"], bbox)] = item["value_text"]
        expected_by_text[(item["page"], item["suffix"], item["key_text"].strip())] = item["value_text"]
    return expected_by_bbox, expected_by_text


def normalize_text(text):
    if text is None:
        return None
    collapsed = " ".join(text.split())
    return collapsed.replace("- ", "-").replace(" -", "-")


docs = {}
for stem in doc_stems:
    entities = load_entities(stem)
    expected_by_bbox, expected_by_text = load_expected(stem)
    docs[stem] = (entities, expected_by_bbox, expected_by_text)

## 3. Asignación voraz global (misma lógica que `_build_form`)

Se reimplementa solo el bucle de generación de candidatos y asignación golosa (para poder
inspeccionar, clave por clave, tanto los casos asignados como los que quedan sin valor),
llamando siempre a `extractor._score_value_candidate`, el método real parametrizado
arriba.

In [ ]:
def assign_values(extractor, entities):
    keys = [e for e in entities if e["label"].startswith(extractor.FIELD_KEY_PREFIX)]
    values = [e for e in entities if e["label"].startswith(extractor.FIELD_VALUE_PREFIX)]

    candidates_by_key = []
    for key in keys:
        suffix = key["label"][len(extractor.FIELD_KEY_PREFIX):]
        expected_label = extractor.FIELD_VALUE_PREFIX + suffix
        key_candidates = []
        for value_idx, value in enumerate(values):
            if value["page"] != key["page"] or value["label"] != expected_label:
                continue
            score = extractor._score_value_candidate(key["bbox"], value["bbox"])
            if score is None:
                continue
            key_candidates.append({"value_idx": value_idx, "value": value, "score": score})
        key_candidates.sort(key=lambda c: c["score"], reverse=True)
        candidates_by_key.append(key_candidates)

    global_pairs = []
    for key_id, key_candidates in enumerate(candidates_by_key):
        for candidate in key_candidates:
            global_pairs.append((key_id, candidate))
    global_pairs.sort(key=lambda pair: pair[1]["score"], reverse=True)

    assigned_by_key = {}
    used_values = set()
    for key_id, candidate in global_pairs:
        if key_id in assigned_by_key or candidate["value_idx"] in used_values:
            continue
        assigned_by_key[key_id] = candidate
        used_values.add(candidate["value_idx"])

    results = []
    for key_idx, key in enumerate(keys):
        best = assigned_by_key.get(key_idx)
        results.append({
            "page": key["page"],
            "suffix": key["label"][len(extractor.FIELD_KEY_PREFIX):],
            "key_text": key["text"],
            "key_bbox": tuple(key["bbox"]),
            "predicted_value_text": best["value"]["text"] if best is not None else None,
        })
    return results


def evaluate(same_row_bonus):
    extractor = make_extractor(same_row_bonus)
    total = 0
    correct = 0
    doc_exact = 0
    for stem, (entities, expected_by_bbox, expected_by_text) in docs.items():
        results = assign_values(extractor, entities)
        predicted_by_bbox = {r["key_bbox"]: r["predicted_value_text"] for r in results}

        doc_total = 0
        doc_correct = 0
        for r in results:
            bbox_key = (r["page"], r["suffix"], r["key_bbox"])
            if bbox_key in expected_by_bbox:
                expected_text = expected_by_bbox[bbox_key]
            else:
                text_key = (r["page"], r["suffix"], r["key_text"].strip())
                expected_text = expected_by_text.get(text_key)
            if expected_text is None:
                continue
            doc_total += 1
            if normalize_text(expected_text) == normalize_text(r["predicted_value_text"]):
                doc_correct += 1

        total += doc_total
        correct += doc_correct
        if doc_correct == doc_total:
            doc_exact += 1

    return doc_exact, correct, total

## 4. Barrido de `q(k,v)`

In [ ]:
bonus_values = [1.0, 1.05, 1.1, 1.15, 1.2, 1.25, 1.3, 1.4, 1.5, 1.75, 2.0]
rows = []
for bonus in bonus_values:
    doc_exact, correct, total = evaluate(bonus)
    rows.append({
        "q(k,v)": bonus,
        "documentos_exactos": doc_exact,
        "parejas_correctas": correct,
        "parejas_totales": total,
    })

bonus_df = pd.DataFrame(rows)
bonus_df

,"q(k,v)",documentos_exactos,parejas_correctas,parejas_totales
0,1.00,0,790,834
1,1.05,22,828,834
2,1.10,25,834,834
3,1.15,25,834,834
4,1.20,25,834,834
5,1.25,24,833,834
6,1.30,24,833,834
7,1.40,24,833,834
8,1.50,24,833,834
9,1.75,24,833,834


### Conclusión

(completar tras revisar el barrido de la celda anterior)